# Attitude agility — roll

A worked example of the `quicksat` agility budget about the **roll** axis. It shows what the reaction wheels allow, and how long a rest-to-rest slew takes. Then it checks the slews that the mission needs, from the requirements file. It covers both the nominal case with four wheels and the case with one wheel failed.

Pitch uses the same calculation, and has its own notebook, [`agility_pitch.ipynb`](agility_pitch.ipynb).

`docs/sizing/agility_ref.ipynb` explains *why* each step works the way it does. It covers the two kinds of inertia case, where the wheel projection comes from, and why one failed wheel halves the axis. `docs/sizing/mission_ref.ipynb` covers the mission that gives the ground track.

In [1]:
# Change log. LAST_CHANGE and CHANGE_NOTE are typed in by hand: update them whenever
# the inputs move -- a wheel changed, an inertia case remeasured, a settling time
# renegotiated -- so that the figures below can be read against what produced them.
# The run time is recorded automatically, and says how stale the outputs stored in
# this notebook are relative to that last change.
from datetime import datetime

LAST_CHANGE = "2026-10-21"
CHANGE_NOTE = "first_guess is the selected case; the slews to check come from requirements.yaml."

print(f"last change  {LAST_CHANGE}")
print(f"             {CHANGE_NOTE}")
print(f"last run     {datetime.now().astimezone():%Y-%m-%d %H:%M %Z}")

last change  2026-10-21
             first_guess is the selected case; the slews to check come from requirements.yaml.
last run     2026-10-22 23:51 CEST


In [2]:
import sys
from pathlib import Path

import pandas as pd

# quicksat is installed with pip. In a clone of the repository, use the package
# beside sample/ instead, so that it runs without installing. Either way the
# notebook runs in its own folder, and reads data/ beside it.
repo = Path.cwd().parents[1]
if (repo / "quicksat" / "__init__.py").exists():
    sys.path.insert(0, str(repo))

from quicksat import u
from quicksat.agility.budget import AgilityBudget, Axis
from quicksat.mass.budget import MassBudget
from quicksat.utils.checks import tabulate as tabulate_checks
from quicksat.utils.mission import Mission

pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

## The inputs

There are four files. The agility config holds the inertia cases, the case it selects, the wheels and the settling time. The shared mission gives the ground track speed. The mass budget gives the spacecraft's mass. The shared requirements give the slews that the mission needs. The mission is loaded once and passed on, rather than read again, as in the other budgets.

The agility config holds **no mass**. The mass budget holds it, and a second copy here could come to disagree with it. For the same reason, the altitude is only in `mission.yaml`. The config holds no target duration either. How long a slew may take is a requirement placed on a spacecraft, not a property of it. So it is in `requirements.yaml`.

In [3]:
DATA = Path("data")
mission = Mission.from_yaml_file(DATA / "mission.yaml")
mass_data = MassBudget.from_csv(DATA / "equipment.csv", DATA / "mass_budget_config.yaml")

roll = AgilityBudget.from_yaml_file(
    DATA / "agility_config.yaml",
    DATA / "mission.yaml",
    Axis.ROLL,
    mass_budget=mass_data,
    requirements_path=DATA / "requirements.yaml",
)

config = roll.config
print(f"mission       {mission.altitude:.0f}, ground track {mission.ground_track_speed:.3f}")
print(f"wheels        {config.wheels.count} at {config.wheels.elevation} elevation")
print(f"settling      {config.settling_time}")
print(f"inertia cases {', '.join(config.inertia_cases)}")
print()
print(f"flying        '{roll.case_name}', the selected case -- an envelope, so the mass comes from the")
print(f"              mass budget at {roll.case.propellant:.0f} propellant: {roll.mass:.2f}")
print(f"inertia       {roll.inertia:.1f}")

mission       500 km, ground track 7.059 km / s
wheels        4 at 26.5 deg elevation
settling      20.0 s
inertia cases first_guess, measured_bol, measured_eol

flying        'first_guess', the selected case -- an envelope, so the mass comes from the
              mass budget at 100 % propellant: 470.62 kg
inertia       262.3 m2 kg


## What the wheels can apply about the axis

The pyramid's symmetry axis is along yaw. So roll and pitch both lie in its base plane, and get the same fraction of each wheel. That fraction is the projection. `cos(elevation)` projects the wheel into the plane, and `cos(45°)` projects it onto the axis within the plane.

Only part of each wheel is available. The momentum use factor and the torque derating are policy, not hardware. The momentum held back stores disturbances, and the torque held back keeps control authority during the slew.

In [4]:
print(f"usable per wheel  {roll.usable_momentum_per_wheel:.3f}"
      f"  and {roll.usable_torque_per_wheel:.3f}")
print(f"projection        {roll.projection.value:.4f}")
print()
print(f"about roll        {roll.axis_momentum():.3f}  {roll.axis_torque():.4f}")
print(f"about yaw         {roll.yaw_momentum:.2f}   -- the weak axis under this")
print("                                    mounting; no slew case here, but it is")
print("                                    what a yaw manoeuvre lives within")

usable per wheel  1.332 N m s  and 0.150 N m
projection        0.6328

about roll        3.372 N m s  0.3797 N m
about yaw         2.38 N m s   -- the weak axis under this
                                    mounting; no slew case here, but it is
                                    what a yaw manoeuvre lives within


## The two limits, and where they swap

Momentum sets the highest rate at which the spacecraft can turn. Torque sets how quickly it reaches that rate. Below the crossover angle, the wheels never saturate, and the slew is **torque limited**. Its rate profile is a triangle: the body accelerates, then decelerates. Above the crossover, the wheels saturate, and the slew coasts at the maximum rate. It is then **momentum limited**, and the profile is a trapezoid.

Which of the two limits a slew is a useful output. For a torque limited slew, more torque would shorten it. For a momentum limited slew, it would not.

In [5]:
print(f"max rate          {roll.max_rate():.4f}")
print(f"max acceleration  {roll.max_acceleration():.5f}")
print(f"crossover angle   {roll.crossover_angle():.2f}")
print()
for angle in (5 * u.deg, 90 * u.deg):
    print(f"  {angle:4.0f} is {roll.profile(angle).value:11s} limited by "
          f"{'torque' if angle <= roll.crossover_angle() else 'momentum'}")

max rate          0.7366 deg / s
max acceleration  0.08295 deg / s2
crossover angle   6.54 deg

     5 deg is triangular  limited by torque
    90 deg is trapezoidal limited by momentum


## How long a slew takes

There is one row per angle. It gives the slew time, the slew time plus settling, the peak rate, and the fraction of the wheel momentum used. Momentum used reaches 100% for every slew past the crossover, which is what momentum limited means. Below the crossover, the rest is spare momentum, which a larger slew would use.

In [6]:
roll.slew_table()

,angle,profile,slew_time,total_time,peak_rate,momentum_used
0,5.00,triangular,15.53,35.53,0.64,0.87
1,10.00,trapezoidal,22.46,42.46,0.74,1.00
2,15.00,trapezoidal,29.24,49.24,0.74,1.00
3,20.00,trapezoidal,36.03,56.03,0.74,1.00
4,40.00,trapezoidal,63.19,83.19,0.74,1.00
5,45.00,trapezoidal,69.97,89.97,0.74,1.00
6,60.00,trapezoidal,90.34,110.34,0.74,1.00
7,90.00,trapezoidal,131.07,151.07,0.74,1.00


## Does it fit?

The requirements file lists the slews that the mission needs, each with the time it may take, settling included. `checks()` checks each slew about this axis, for the selected inertia case and with every wheel working. `tabulate()` from `quicksat.utils.checks` lays the checks out. The Sizing Editor shows the same checks, from the same code.

Different operations may ask different times of the same spacecraft. So `time_margin()` and `achievable_angle()` also take any time as an argument. Asked the other way round, the budget says how far the spacecraft could turn in that time.

The ground track keeps moving during a slew, and the payload images none of it. The shared mission's ground track speed turns the slew time into that distance on the ground. A payload operator measures a slew by this distance.

In [7]:
tabulate_checks(roll.checks())

check,description,design,limit,spare,margin,verdict
roll_40,"A 40° roll slew within 90 s, settling included, for the inertia case first_guess",83.19 s,≤ 90.00 s,+6.81 s,+8.2 %,PASS


In [8]:
for target in (90 * u.s, 113 * u.s):
    print(f"in {target:.0f}:  largest slew {roll.achievable_angle(target):.1f}, "
          f"while the ground track runs {roll.ground_distance(target):.0f}")

in 90 s:  largest slew 45.0 deg, while the ground track runs 635 km
in 113 s:  largest slew 62.0 deg, while the ground track runs 798 km


## Which spacecraft are we slewing?

The config names three inertia cases, and selects `first_guess`. `first_guess` estimates the inertia from a box, and takes its mass from the mass budget. So it changes whenever the equipment list does. `measured_bol` and `measured_eol` state an inertia directly, as a mass properties report gives it, and need no mass at all.

The check uses the selected case only. The others are kept for comparison, such as the spacecraft at other stages of the mission. A budget for another case takes its name, as below. We chose the names, and nothing in the code depends on them. What matters is that every number can be traced to the spacecraft case that produced it.

In [9]:
cases = []
for name in config.inertia_cases:
    case = AgilityBudget(
        config, mission, Axis.ROLL, name, mass_budget=mass_data,
        requirements=roll.requirements,
    )
    (check,) = case.checks()
    cases.append({
        "case": name + (" (selected)" if name == config.selected_case else ""),
        "shape": "envelope" if case.case.is_envelope else "stated",
        "inertia": case.inertia.value,
        "max_rate": case.max_rate().value,
        "slew_40_deg": case.slew_time(40 * u.deg).value,
        "margin_in_90_s": check.margin.to_value("percent"),
    })

pd.DataFrame(cases).style.format(
    {"inertia": "{:,.1f}", "max_rate": "{:,.4f}",
     "slew_40_deg": "{:,.2f}", "margin_in_90_s": "{:+.1f} %"}
).hide(  # pyright: ignore[reportAttributeAccessIssue]
    axis="index"
)

case,shape,inertia,max_rate,slew_40_deg,margin_in_90_s
first_guess (selected),envelope,262.3,0.7366,63.19,+8.2 %
measured_bol,stated,264.7,0.7298,63.69,+7.5 %
measured_eol,stated,250.0,0.7727,60.65,+11.6 %


The spread between the cases is worth reading. `measured_eol` is the same spacecraft with its tanks empty. It is lighter, so it turns faster, and reaches a larger angle in the same time. Agility is one of the few budgets that *improves* over a mission.

## The whole thing, in one table

`tabulated_agility()` lays out the slew table as a document. Given a target duration, it also checks each angle against it, and marks the slews that do not fit. Without one, those columns are left out, because there is nothing to check against.

In [10]:
print(f"inertia case  '{roll.case_name}'  {roll.inertia:.1f}")
print()
roll.tabulated_agility(target_duration=90 * u.s)

inertia case  'first_guess'  262.3 m2 kg



Slew [deg],Limited by,Slew [s],With settling [s],Peak rate [deg/s],Momentum used,Time margin,
5,torque,15.5,35.5,0.6440,87.4%,+153.3%,PASS
10,momentum,22.5,42.5,0.7366,100.0%,+112.0%,PASS
15,momentum,29.2,49.2,0.7366,100.0%,+82.8%,PASS
20,momentum,36.0,56.0,0.7366,100.0%,+60.6%,PASS
40,momentum,63.2,83.2,0.7366,100.0%,+8.2%,PASS
45,momentum,70.0,90.0,0.7366,100.0%,+0.0%,PASS
60,momentum,90.3,110.3,0.7366,100.0%,-18.4%,FAILS
90,momentum,131.1,151.1,0.7366,100.0%,-40.4%,FAILS


## One wheel failed

With one of the four wheels failed, three remain. Only two of them can run at full torque if the net momentum in the base plane is to stay zero. So both the momentum and the torque about the axis halve. The rate and the acceleration halve with them.

This is the same pyramid with one wheel failed, not a mounting designed for three wheels. The geometry and the projection do not change. Only the number of usable wheels drops.

In [11]:
nominal = roll.slew_table()[["angle", "slew_time"]]
degraded = roll.slew_table(degraded=True)[["angle", "slew_time"]]

comparison = nominal.merge(degraded, on="angle", suffixes=("_4_wheels", "_3_wheels"))
comparison["penalty"] = (
    comparison["slew_time_3_wheels"] / comparison["slew_time_4_wheels"] - 1
)
comparison.style.format(
    {"angle": "{:,.0f}", "slew_time_4_wheels": "{:,.1f}",
     "slew_time_3_wheels": "{:,.1f}", "penalty": "{:+.0%}"}
).hide(  # pyright: ignore[reportAttributeAccessIssue]
    axis="index"
)

angle,slew_time_4_wheels,slew_time_3_wheels,penalty
5,15.5,22.5,+45%
10,22.5,36.0,+60%
15,29.2,49.6,+70%
20,36.0,63.2,+75%
40,63.2,117.5,+86%
45,70.0,131.1,+87%
60,90.3,171.8,+90%
90,131.1,253.3,+93%
